In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.3 MB/s eta 0:00:00


In [2]:
#  Scenario: Package Delivery in a City
# Imagine you run a delivery company in a huge city with 100,000 houses. Each house has a
#  unique “address fingerprint” (like the 384‑dimensional vectors in your code).
# Now, when a customer places an order, you need to quickly find the nearest warehouse or
#  delivery hub to their house. Instead of checking every single house one by one (which would
# be very slow), you use smart strategies:

# 🗺️ HNSW (Hierarchical Navigable Small World)
# - Think of it like a layered city map:
# - Top layer = highways (coarse overview).
# - Middle layer = main roads.
# - Bottom layer = all streets and houses.
# - To find the nearest warehouse, you start at the highway level, zoom into the right district,
# then finally navigate down to the exact street.
# - This is what IndexHNSWFlat does: it builds a graph of connections so you can “hop” quickly
# to the right neighborhood.

# 🏘️ IVF (Inverted File Index)
# - Imagine dividing the city into 256 neighborhoods (Voronoi cells).
# - First, figure out which neighborhood the customer’s house belongs to.
# - Then, only search inside that neighborhood instead of the whole city.
# - This is what IndexIVFFlat does: it clusters vectors and searches only in the most relevant
# clusters.
# - nprobe = 10 means you don’t just check one neighborhood, but the 10 most likely ones —
#  balancing speed and accuracy.

# ⚡ In Plain Words
# - HNSW = zooming in layer by layer on a city map.
# - IVF = splitting the city into neighborhoods and searching locally.
# - Both methods help you find the closest match fast without scanning all 100,000 houses.


import faiss
import numpy as np

d = 384       # embedding dimension
n = 100_000   # number of vectors

# ── HNSW index ──────────────────────────────
index = faiss.IndexHNSWFlat(d, 32)  # M=32 neighbours
index.hnsw.efConstruction = 200     # build quality
index.hnsw.efSearch = 100           # query quality

vectors = np.random.randn(n, d).astype("float32")
faiss.normalize_L2(vectors)         # L2 normalize
index.add(vectors)                  # add all vectors

query = np.random.randn(1, d).astype("float32")
faiss.normalize_L2(query)
D, I = index.search(query, k=10)    # top-10
print("Indices:", I[0])             # [2341, 8712, ...]
print("Distances:", D[0])           # cosine scores

# ── IVF index (faster for large scale) ──────
nlist = 256   # number of Voronoi cells
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
index_ivf.train(vectors)
index_ivf.add(vectors)
index_ivf.nprobe = 10               # cells to search

Indices: [51120 46939 32912  5453 25185  4056 31163 36230 74413 72506]
Distances: [1.5798643 1.6101964 1.639244  1.6416806 1.6449022 1.6547891 1.6554475
 1.6602712 1.6641009 1.6642836]


In [3]:
!pip install chromadb
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [4]:
# Scenario: Student Course Recommendations
# Imagine you’re running an online learning platform. Students type queries like “beginner-friendly Python tutorials”,
# but your catalog has courses titled “Introduction to Programming with Python”.
# Traditional keyword search might miss the match, but embeddings + vector search (Chroma locally or Pinecone in the cloud)
#  can connect them semantically.

# 🧩 Teaching Exercise Flow
# - Documents (Courses)
# - "Python is a high-level programming language"
# - "Machine learning models need training data"
# - "Dogs are loyal and friendly animals"
# - "Cats are independent and curious pets"
# - Embedding Model
# - Converts each course description into a vector (numbers capturing meaning).
# - Indexing
# - Chroma (local): Great for classroom demos, no API key needed.
# - Pinecone (cloud): Scales to billions of items, perfect for production.
# - Query
# - Student asks: “What animals make good companions?”
# - Embedding of query is compared against course embeddings.
# - Results
# - Top matches:
# - Cats are independent and curious pets
# - Dogs are loyal and friendly animals

# OPTION A: Chroma (local, no API key needed)
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create Chroma client and collection
client = chromadb.Client()  # or chromadb.PersistentClient(path="./db")
collection = client.create_collection("docs")

# Sample documents
documents = [
    "Python is a high-level programming language",
    "Machine learning models need training data",
    "Dogs are loyal and friendly animals",
    "Cats are independent and curious pets",
]

# Encode documents into embeddings
embeddings = model.encode(documents).tolist()

# Add to Chroma collection
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc{i}" for i in range(len(documents))],
    metadatas=[{"source": "demo", "idx": i} for i in range(len(documents))]
)

# Query with semantic search
query = "What animals make good companions?"
q_emb = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

# Print results
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[0.806] Dogs are loyal and friendly animals
[1.224] Cats are independent and curious pets
[1.782] Python is a high-level programming language
